# Bhutanese Code-Mixed Smart Home Assistant — DistilBERT Intent Classification

## NLP Pipeline

**Raw dataset → Data audit → Minimal text normalization → Intent label encoding → Leakage-safe train/validation/test split → DistilBERT tokenization → Fine-tuning → Validation-based model selection → Test evaluation → Error analysis → Save model**

This notebook trains **`distilbert-base-multilingual-cased`** for **intent classification**. The model receives only the user's **text command** and predicts the **Intent**.

> **Important:** `Device`, `Control value`, and `Language Type` are metadata used for analysis/evaluation. They are **not input features** to the intent classifier, so the model learns the intent from the utterance itself.


## Run in Colab

1. Select a GPU in **Runtime → Change runtime type**, then choose **Runtime → Run all**.
2. Authorize Google Drive in Section 2. In Section 3, upload your original `distilbert_output` ZIP, select `train.csv`, `validation.csv`, and `test.csv` together, or upload the original Excel dataset.
3. The notebook writes a new run under **My Drive → distilbert_colab_runs**. The final section shows where to download the trained weights and how to try the saved model.

## 1. Setup

Install libraries needed for data handling, Hugging Face training, and evaluation.

In [ ]:
%pip install -q "transformers==5.18.0" datasets accelerate scikit-learn openpyxl

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

In [ ]:
# Check the available hardware
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Configuration

Mount Google Drive and create a separate output folder for this run. Training checkpoints, the prepared splits, and the final model are kept there across Colab sessions.

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

from google.colab import drive, files

drive.mount("/content/drive")

run_name = datetime.now(ZoneInfo("Asia/Thimphu")).strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"/content/drive/MyDrive/distilbert_colab_runs/{run_name}"
MODEL_DIR = os.path.join(OUTPUT_DIR, "distilbert_intent_model")
SPLIT_DIR = os.path.join(OUTPUT_DIR, "splits")
os.makedirs(SPLIT_DIR)

print("Output folder", OUTPUT_DIR)

## 3. Load and audit the dataset

The upload prompt accepts the original output ZIP, the three split CSVs selected together, or the original Excel workbook. The saved splits keep their original training, validation, and test assignments.

- `Transcript` (Excel) or `text` (CSV) is the input.
- `Intent` is the target label.
- `Language Type`, `Device`, and `Control value` are metadata for evaluation and demonstration.

In [ ]:
from io import BytesIO
from zipfile import ZipFile

split_files = {"train": "train.csv", "validation": "validation.csv", "test": "test.csv"}
print("Upload the original ZIP, all three split CSVs together, or the Excel dataset.")
uploaded = files.upload()
saved_splits = None

if set(split_files.values()).issubset(uploaded):
    saved_splits = {
        split: pd.read_csv(BytesIO(uploaded[filename]))
        for split, filename in split_files.items()
    }
elif len(uploaded) == 1:
    filename, payload = next(iter(uploaded.items()))
    if filename.lower().endswith(".xlsx"):
        df = pd.read_excel(BytesIO(payload))
    elif filename.lower().endswith(".zip"):
        saved_splits = {}
        with ZipFile(BytesIO(payload)) as archive:
            for split, csv_name in split_files.items():
                matches = [member for member in archive.namelist() if Path(member).name == csv_name]
                if len(matches) != 1:
                    raise ValueError(f"The ZIP must contain exactly one {csv_name}.")
                with archive.open(matches[0]) as csv_file:
                    saved_splits[split] = pd.read_csv(csv_file)
    else:
        raise ValueError("Upload a ZIP, an Excel workbook, or all three split CSVs together.")
else:
    raise ValueError("Upload a ZIP, an Excel workbook, or all three split CSVs together.")

if saved_splits is not None:
    for split, table in saved_splits.items():
        required = {"text", "label", "Intent", "Language Type", "Device", "Control value"}
        missing = required.difference(table.columns)
        if missing or table.empty:
            raise ValueError(f"Invalid {split} split. Missing columns: {sorted(missing)}. Rows: {len(table)}.")
    df = pd.concat(saved_splits.values(), ignore_index=True).rename(columns={"text": "Transcript"})

print("Dataset shape", df.shape)
print("Columns", df.columns.tolist())
display(df.head())

In [ ]:
required_columns = ["Transcript", "Intent", "Language Type", "Device", "Control value"]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Missing values:")
print(df[required_columns].isnull().sum())


In [ ]:
print("Intent distribution:")
print(df["Intent"].value_counts())

print("\nLanguage distribution:")
print(df["Language Type"].value_counts())

print("\nDuplicate transcript rows:", df["Transcript"].duplicated().sum())


In [ ]:
# Inspect representative commands
pd.set_option("display.max_colwidth", None)

display(
    df[["Transcript", "Language Type", "Intent", "Device", "Control value"]]
    .sample(min(20, len(df)), random_state=42)
)


## 4. Minimal text normalization

DistilBERT uses a pretrained subword tokenizer. Therefore, do **not** apply traditional NLP preprocessing such as:

- stop-word removal
- stemming
- lemmatization
- removing punctuation
- converting everything to lowercase

Those operations can remove information that is useful for intent classification, especially in short code-mixed commands.

Only whitespace is normalized here.

In [ ]:
def clean_text(text):
    return " ".join(str(text).split())

df = df.dropna(subset=["Transcript", "Intent"]).copy()
df["text"] = df["Transcript"].map(clean_text)
df = df.loc[df["text"].ne("")].copy()

print("Samples after basic cleaning", len(df))
display(df[["Transcript", "text"]].sample(min(10, len(df)), random_state=42))

### Check repeated utterances for data leakage

Your dataset may contain multiple recordings of the same transcript. Those rows should **not** be allowed to appear in different splits because the model could effectively see the same wording during training and testing.

We therefore keep repeated rows, but use the normalized transcript as a **group** during splitting so all recordings of the same utterance stay in the same split.

In [ ]:
# Check whether a transcript is associated with more than one intent.
conflicting_groups = (
    df.groupby("text")["Intent"]
      .nunique()
      .loc[lambda s: s > 1]
)

print("Utterances with conflicting intent labels:", len(conflicting_groups))

if len(conflicting_groups) > 0:
    print("\nExamples:")
    display(
        df[df["text"].isin(conflicting_groups.index)]
        [["text", "Intent"]]
        .drop_duplicates()
        .sort_values("text")
        .head(30)
    )
    raise ValueError(
        "Some identical utterances have different intent labels. "
        "Resolve these labels before training."
    )


## 5. Encode intent labels

The model requires integer class IDs. The mapping is saved so the same labels can be used later by the application.

In [ ]:
intent_classes = sorted(df["Intent"].unique())

label2id = {label: idx for idx, label in enumerate(intent_classes)}
id2label = {idx: label for label, idx in label2id.items()}

df["label"] = df["Intent"].map(label2id).astype(int)

print("Number of intent classes:", len(intent_classes))
print("\nLabel mapping:")
for idx, label in id2label.items():
    print(f"{idx}: {label}")

print("\nMissing encoded labels:", df["label"].isna().sum())


## 6. Train / validation / test split

The Excel workflow creates an approximately **70% training, 15% validation, and 15% testing** split grouped by normalized transcript. The CSV workflow reuses the original assignments. Both workflows check that identical normalized commands appear in only one split.

- **Train** is used to fine-tune DistilBERT.
- **Validation** is used during training to select the best checkpoint.
- **Test** is kept for final evaluation.

In [ ]:
if saved_splits is None:
    # First split: 70% train, 30% temporary
    gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)

    train_idx, temp_idx = next(
        gss_1.split(
            df,
            y=df["Intent"],
            groups=df["text"]
        )
    )

    train_df = df.iloc[train_idx].copy()
    temp_df = df.iloc[temp_idx].copy()

    # Second split: half of the 30% temporary set for validation and half for test
    # This gives approximately 70/15/15 overall.
    gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)

    val_idx, test_idx = next(
        gss_2.split(
            temp_df,
            y=temp_df["Intent"],
            groups=temp_df["text"]
        )
    )

    val_df = temp_df.iloc[val_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
else:
    def prepare_saved_split(table):
        table = table.copy()
        table["text"] = table["text"].fillna("").map(clean_text)
        if table["text"].eq("").any() or not table["Intent"].map(label2id).eq(table["label"]).all():
            raise ValueError("Saved splits must contain commands and match the intent label mapping.")
        return table

    train_df = prepare_saved_split(saved_splits["train"])
    val_df = prepare_saved_split(saved_splits["validation"])
    test_df = prepare_saved_split(saved_splits["test"])

print(f"Train:      {len(train_df)} ({len(train_df)/len(df):.1%})")
print(f"Validation: {len(val_df)} ({len(val_df)/len(df):.1%})")
print(f"Test:       {len(test_df)} ({len(test_df)/len(df):.1%})")

In [ ]:
# Verify that no normalized transcript occurs in more than one split
train_groups = set(train_df["text"])
val_groups = set(val_df["text"])
test_groups = set(test_df["text"])

print("Train ∩ Validation:", len(train_groups & val_groups))
print("Train ∩ Test:", len(train_groups & test_groups))
print("Validation ∩ Test:", len(val_groups & test_groups))

assert len(train_groups & val_groups) == 0
assert len(train_groups & test_groups) == 0
assert len(val_groups & test_groups) == 0


In [ ]:
print("Intent distribution by split:")
split_summary = pd.DataFrame({
    "Train": train_df["Intent"].value_counts(),
    "Validation": val_df["Intent"].value_counts(),
    "Test": test_df["Intent"].value_counts(),
}).fillna(0).astype(int)

display(split_summary)

print("\nLanguage distribution by split:")
language_summary = pd.DataFrame({
    "Train": train_df["Language Type"].value_counts(),
    "Validation": val_df["Language Type"].value_counts(),
    "Test": test_df["Language Type"].value_counts(),
}).fillna(0).astype(int)

display(language_summary)


## 7. Save the prepared splits and label mapping

Saving the splits is optional for training, but useful for reproducibility and later analysis.

In [ ]:
# Keep only columns needed for the model plus metadata for evaluation.
model_columns = [
    "text",
    "label",
    "Intent",
    "Language Type",
    "Device",
    "Control value"
]

train_df[model_columns].to_csv(os.path.join(SPLIT_DIR, "train.csv"), index=False)
val_df[model_columns].to_csv(os.path.join(SPLIT_DIR, "validation.csv"), index=False)
test_df[model_columns].to_csv(os.path.join(SPLIT_DIR, "test.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "label2id.json"), "w") as f:
    json.dump(label2id, f, indent=4)

with open(os.path.join(OUTPUT_DIR, "id2label.json"), "w") as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=4)

print("Prepared data and label mappings saved.")


## 8. Convert to Hugging Face Datasets

Hugging Face `DatasetDict` works directly with the Transformers `Trainer`.

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[model_columns], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[model_columns], preserve_index=False),
    "test": Dataset.from_pandas(test_df[model_columns], preserve_index=False),
})

dataset


## 9. DistilBERT tokenizer

Because the project contains English and Bhutanese code-mixed commands, use the **multilingual cased DistilBERT** checkpoint:

`distilbert-base-multilingual-cased`

In [ ]:
MODEL_NAME = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 64

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_token_type_ids=False
    )

tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True
)

display(tokenized_dataset["train"][0])

In [ ]:
# Inspect one example so the tokenization step is transparent.
sample_text = train_df.iloc[0]["text"]

print("Original text:")
print(sample_text)

print("\nTokens:")
print(tokenizer.tokenize(sample_text))

print("\nToken IDs:")
print(tokenizer(sample_text)["input_ids"])


## 10. Dynamic padding

Dynamic padding pads each batch only to the longest sequence in that batch. This is more efficient than padding the entire dataset to a fixed length.

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


## 11. Load the pretrained DistilBERT classifier

The pretrained multilingual DistilBERT encoder is combined with a classification head whose output size equals the number of intent classes.

This is **fine-tuning**, not training DistilBERT from scratch.

In [ ]:
set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(intent_classes),
    id2label=id2label,
    label2id=label2id
)

print("Model loaded.")

## 12. Evaluation metrics

For multi-class intent classification, report:

- **Accuracy**
- **Weighted Precision**
- **Weighted Recall**
- **Weighted F1-score**

The validation F1-score is used to select the best checkpoint during training.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


## 13. Training configuration

The settings below are appropriate starting values for a small intent dataset.

The model is evaluated after every epoch and the checkpoint with the highest validation weighted F1 is restored before final testing.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=10,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,
    logging_strategy="epoch",

    report_to="none",
    fp16=torch.cuda.is_available(),

    seed=42
)


## 14. Create the Trainer

The Trainer receives:

**tokenized train set + tokenized validation set + DistilBERT + tokenizer/collator + evaluation metrics**.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)


## 15. Train DistilBERT

Only the training set is used for parameter updates. The validation set is used for evaluation/checkpoint selection.

In [ ]:
train_result = trainer.train()
print("Training completed.")
print("Best checkpoint", trainer.state.best_model_checkpoint)
print("Best validation weighted F1", trainer.state.best_metric)

## 16. Final test evaluation

The test set is evaluated **after training is complete**. It was not used for model selection.

In [ ]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"]
)

print("Final Test Results")
print(f"Accuracy : {test_results['eval_accuracy']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall   : {test_results['eval_recall']:.4f}")
print(f"F1       : {test_results['eval_f1']:.4f}")


## 17. Detailed classification report

This shows precision, recall, and F1 for every intent rather than only the overall weighted scores.

In [ ]:
predictions = trainer.predict(tokenized_dataset["test"])

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(len(intent_classes))),
        target_names=[id2label[i] for i in range(len(intent_classes))],
        digits=4,
        zero_division=0
    )
)


## 18. Confusion matrix

The confusion matrix shows which intents are being confused with each other.

In [ ]:
cm = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=list(range(len(intent_classes)))
)

fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(len(intent_classes)))
ax.set_yticks(range(len(intent_classes)))

ax.set_xticklabels(intent_classes, rotation=45, ha="right")
ax.set_yticklabels(intent_classes)

ax.set_xlabel("Predicted Intent")
ax.set_ylabel("True Intent")
ax.set_title("DistilBERT Intent Classification Confusion Matrix")

for i in range(len(intent_classes)):
    for j in range(len(intent_classes)):
        ax.text(
            j, i, cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 19. English vs Code-Mixed evaluation

`Language Type` is **not given to the model**. It is used only after testing to understand whether performance differs between English and code-mixed commands.

In [ ]:
test_results_df = test_df.copy()

test_results_df["true_label"] = true_labels
test_results_df["predicted_label"] = predicted_labels

test_results_df["true_intent"] = test_results_df["true_label"].map(id2label)
test_results_df["predicted_intent"] = test_results_df["predicted_label"].map(id2label)

def evaluate_subset(subset_df, name):
    precision, recall, f1, _ = precision_recall_fscore_support(
        subset_df["true_label"],
        subset_df["predicted_label"],
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(
        subset_df["true_label"],
        subset_df["predicted_label"]
    )

    print(f"{name}")
    print(f"Samples   : {len(subset_df)}")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1        : {f1:.4f}")
    print()

    return {
        "samples": len(subset_df),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

english_metrics = evaluate_subset(
    test_results_df[test_results_df["Language Type"] == "English"],
    "English"
)

codemix_metrics = evaluate_subset(
    test_results_df[test_results_df["Language Type"] == "CodeMix"],
    "Code-Mixed"
)


## 20. Error analysis

Inspect incorrect predictions to identify patterns such as:

- similar commands between intents
- ambiguous wording
- code-mixed expressions
- insufficient training examples
- errors between closely related intents


In [ ]:
incorrect = test_results_df[
    test_results_df["true_label"] != test_results_df["predicted_label"]
]

print("Incorrect predictions:", len(incorrect))

display(
    incorrect[
        [
            "text",
            "Language Type",
            "true_intent",
            "predicted_intent",
            "Device",
            "Control value"
        ]
    ].head(30)
)


## 21. Save the final trained model

The model, tokenizer, and configuration are saved together so the same intent classifier can later be loaded by the smart-home application.

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

weights_path = Path(MODEL_DIR) / "model.safetensors"
if not weights_path.is_file():
    raise FileNotFoundError(f"The trained weights were not saved to {weights_path}.")

print("Saved model weights", weights_path)
print(f"File size {weights_path.stat().st_size / 1_000_000:.1f} MB")

In [ ]:
final_results = {
    "model": MODEL_NAME,
    "num_classes": len(intent_classes),
    "train_samples": len(train_df),
    "validation_samples": len(val_df),
    "test_samples": len(test_df),

    "test_accuracy": float(test_results["eval_accuracy"]),
    "test_precision_weighted": float(test_results["eval_precision"]),
    "test_recall_weighted": float(test_results["eval_recall"]),
    "test_f1_weighted": float(test_results["eval_f1"]),

    "english": {k: float(v) for k, v in english_metrics.items()},
    "code_mixed": {k: float(v) for k, v in codemix_metrics.items()}
}

with open(os.path.join(OUTPUT_DIR, "final_results.json"), "w") as f:
    json.dump(final_results, f, indent=4)

print(json.dumps(final_results, indent=4))


## 22. Device identification (rule-based)

DistilBERT in this notebook is responsible **only for intent classification**. Device identification is handled separately because the project currently has a small, fixed device vocabulary: **Light, Fan, AC, and Curtain**.

The device name is extracted from the user command using a transparent keyword/entity-matching layer. `Device` is therefore **not an input feature to DistilBERT** and is not used to train the intent classifier.

In [ ]:
DEVICE_KEYWORDS = {
    "Light": ["light", "lights", "lamp", "bulb"],
    "Fan": ["fan", "fans"],
    "AC": ["ac", "air conditioner", "aircon", "air conditioning"],
    "Curtain": ["curtain", "curtains"]
}

def normalize_for_device(text):
    return " ".join(str(text).lower().split())

def detect_device(text):
    text = normalize_for_device(text)

    # Check multi-word terms first.
    for device, keywords in DEVICE_KEYWORDS.items():
        for keyword in sorted(keywords, key=len, reverse=True):
            if keyword in text:
                return device

    return "Unknown"

test_results_df["predicted_device"] = test_results_df["text"].apply(detect_device)

print(test_results_df[["text", "Device", "predicted_device"]].head(20))


## 23. Evaluate device identification

Device identification is evaluated separately from DistilBERT. This tells us how well the rule-based device extractor recognizes the four supported bedroom devices from the test commands.

In [ ]:
def normalize_device_label(value):
    value = str(value).strip().lower()

    aliases = {
        "light": "Light",
        "lights": "Light",
        "lamp": "Light",
        "bulb": "Light",
        "fan": "Fan",
        "fans": "Fan",
        "ac": "AC",
        "air conditioner": "AC",
        "aircon": "AC",
        "air conditioning": "AC",
        "curtain": "Curtain",
        "curtains": "Curtain"
    }
    return aliases.get(value, str(value).strip())

test_results_df["true_device"] = test_results_df["Device"].apply(normalize_device_label)

device_accuracy = accuracy_score(
    test_results_df["true_device"],
    test_results_df["predicted_device"]
)

print(f"Device identification accuracy: {device_accuracy:.4f}")

print("\nDevice classification report:")
print(
    classification_report(
        test_results_df["true_device"],
        test_results_df["predicted_device"],
        labels=["Light", "Fan", "AC", "Curtain"],
        digits=4,
        zero_division=0
    )
)


## 24. Control-value extraction

`Control value` is also handled outside the intent classifier. For the current MVP, numeric values can be extracted from the command for settings such as temperature or percentage. The predicted intent determines how that value should be interpreted by the application.

Examples:

- `set AC to 22 degrees` → value `22`
- `set fan speed to 50 percent` → value `50`
- `increase fan speed` → no numeric value required; the intent is `IncreaseSpeed`


In [ ]:
import re

def extract_control_value(text):
    text = str(text).lower()

    # Temperature, percentage, or other numeric control values.
    match = re.search(r"(?<!\d)(\d+(?:\.\d+)?)(?:\s*°?\s*(?:c|percent|%))?", text)

    if match:
        value = match.group(1)
        return float(value) if "." in value else int(value)

    return None

test_results_df["predicted_control_value"] = test_results_df["text"].apply(extract_control_value)

print(test_results_df[[
    "text", "true_intent", "predicted_device", "predicted_control_value"
]].head(20))


## 25. Build the final structured command

The application can combine the three outputs into one command structure:

**Utterance → Intent (DistilBERT) + Device (rule-based) + Control value (rule-based) → Smart-home action**

In [ ]:
def predict_intent(text, classifier, command_tokenizer):
    inputs = command_tokenizer(
        clean_text(text),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        return_token_type_ids=False,
    )
    inputs = {key: value.to(classifier.device) for key, value in inputs.items()}
    classifier.eval()
    with torch.inference_mode():
        probabilities = classifier(**inputs).logits.softmax(dim=-1)
    confidence, label = probabilities[0].max(dim=-1)
    return classifier.config.id2label[label.item()], confidence.item()


def parse_command(text):
    intent, _ = predict_intent(text, model, tokenizer)
    return {
        "text": text,
        "intent": intent,
        "device": detect_device(text),
        "control_value": extract_control_value(text),
    }


examples = [
    "turn on the bedroom light",
    "fan on bay nang",
    "set the AC to 22 degrees",
    "close the curtain",
]

for example in examples:
    print(parse_command(example))

## 26. Save the complete inference configuration

Save the device vocabulary together with the DistilBERT model so the application can reproduce the same device-identification logic during deployment.

In [ ]:
device_config = {
    "devices": list(DEVICE_KEYWORDS.keys()),
    "device_keywords": DEVICE_KEYWORDS,
    "intent_model": MODEL_NAME,
    "max_length": MAX_LENGTH
}

with open(os.path.join(MODEL_DIR, "device_config.json"), "w") as f:
    json.dump(device_config, f, indent=4)

print("Device configuration saved.")


export_names = ["model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json", "device_config.json"]
missing = [name for name in export_names if not (Path(MODEL_DIR) / name).is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete export in {MODEL_DIR}. Missing {', '.join(missing)}.")

print("Complete inference export", MODEL_DIR)
for name in export_names:
    print(f"  {name}")

## 27. Save final results

The report should distinguish between **intent classification performance** and **device identification performance** because they are separate components of the NLP pipeline.

In [ ]:
final_results = {
    "model": MODEL_NAME,
    "task": "Intent classification",
    "num_intent_classes": len(intent_classes),
    "train_samples": len(train_df),
    "validation_samples": len(val_df),
    "test_samples": len(test_df),

    "intent_test_accuracy": float(test_results["eval_accuracy"]),
    "intent_test_precision_weighted": float(test_results["eval_precision"]),
    "intent_test_recall_weighted": float(test_results["eval_recall"]),
    "intent_test_f1_weighted": float(test_results["eval_f1"]),

    "device_identification_method": "Rule-based keyword/entity matching",
    "device_test_accuracy": float(device_accuracy),

    "english": {k: float(v) for k, v in english_metrics.items()},
    "code_mixed": {k: float(v) for k, v in codemix_metrics.items()}
}

with open(os.path.join(OUTPUT_DIR, "final_results.json"), "w") as f:
    json.dump(final_results, f, indent=4)

print(json.dumps(final_results, indent=4))


# Final NLP Pipeline Summary

**1. Load dataset** → Read the smart-home command dataset.

**2. Audit data** → Check columns, missing values, intent distribution, language distribution, duplicates, and label consistency.

**3. Minimal normalization** → Normalize whitespace only.

**4. Encode intent labels** → Convert intent names to integer class IDs.

**5. Leakage-safe split** → Create or reuse approximately 70/15/15 train/validation/test sets and verify transcript overlap is zero.

**6. Tokenization** → Use `distilbert-base-multilingual-cased`.

**7. Fine-tuning** → Fine-tune DistilBERT for **intent classification only**.

**8. Validation** → Select the best checkpoint using validation weighted F1.

**9. Final intent testing** → Evaluate accuracy, precision, recall, F1, classification report, and confusion matrix on the untouched test set.

**10. Device identification** → Identify Light, Fan, AC, or Curtain using a separate rule-based keyword/entity layer.

**11. Control-value extraction** → Extract numeric values when present.

**12. Structured command** → Combine intent + device + control value for the virtual smart-home controller.

### Final system architecture

`User utterance → DistilBERT → Intent`

`User utterance → Device extractor → Device`

`User utterance → Value extractor → Control value`

`Intent + Device + Control value → Virtual bedroom action`

### Example

`"bedroom light on bay nang"` → **TurnOn + Light + None**

`"turn off the fan"` → **TurnOff + Fan + None**

`"set AC to 22 degrees"` → **SetTemp + AC + 22**

## 28. Demo the exported model

Reload the classifier and tokenizer from the Drive export. Try an English command and a code-mixed command for the presentation.

Download `model.safetensors` directly from the **distilbert_intent_model** folder printed above. For app inference, keep it with the config, tokenizer, and device keyword files listed in Section 26.

In [ ]:
saved_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
saved_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=True,
).eval()

In [ ]:
demo_commands = [
    "bedroom light siwtch on",
    "Can you turn off the fan?",
    "Make the fan speed more",
    "Reduce the fan speed",
    "air conditioner gobay",
    "Ac to 20 degree",
    "AC temperature mangm",
    "Close the bedroom curtains",
    "Curtain open",
    "light on"
]

In [ ]:
for command in demo_commands:
    intent, confidence = predict_intent(command, saved_model, saved_tokenizer)
    print(f"Command    {command}")
    print(f"Intent     {intent}")
    print(f"Confidence {confidence:.2%}")
    print("-" * 60)